In [1]:
# To generate the official Word Document (.docx) on your machine:
# Run: pip install python-docx
# Then execute this script: python generate_docs.py

import docx
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_ALIGN_VERTICAL
from docx.oxml import parse_xml, OxmlElement
from docx.oxml.ns import nsdecls, qn

def create_document():
    doc = Document()

    # Configure Margins (1 inch)
    for section in doc.sections:
        section.top_margin = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin = Inches(1)
        section.right_margin = Inches(1)

    # Styling Palette Constants
    PRIMARY_COLOR = RGBColor(26, 86, 219)    # Royal Blue
    DARK_TEXT = RGBColor(30, 41, 59)         # Slate 800
    MUTED_TEXT = RGBColor(100, 116, 139)     # Slate 500
    CODE_BG = "F1F5F9"                       # Slate 100
    HEADER_BG = "1E293B"                     # Slate 800

    def set_cell_background(cell, fill_hex):
        tcPr = cell._element.get_or_add_tcPr()
        tcPr.append(parse_xml(f'<w:shd {nsdecls("w")} w:fill="{fill_hex}"/>'))

    def set_cell_margins(cell, top=100, bottom=100, left=150, right=150):
        tcPr = cell._element.get_or_add_tcPr()
        tcMar = OxmlElement('w:tcMar')
        for margin, val in [('top', top), ('bottom', bottom), ('left', left), ('right', right)]:
            node = OxmlElement(f'w:{margin}')
            node.set(qn('w:w'), str(val))
            node.set(qn('w:type'), 'dxa')
            tcMar.append(node)
        tcPr.append(tcMar)

    # --- Title Section ---
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run_title = title.add_run("Jenga P2P Marketplace\nM-Pesa Daraja STK Push & Order Lifecycle Documentation")
    run_title.font.name = "Arial"
    run_title.font.size = Pt(20)
    run_title.font.bold = True
    run_title.font.color.rgb = PRIMARY_COLOR

    subtitle = doc.add_paragraph()
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run_sub = subtitle.add_run("Technical Architecture, Asynchronous Handshake & Production Integration Specification\nDate: September 2026 | Version: 1.0.0")
    run_sub.font.name = "Arial"
    run_sub.font.size = Pt(10)
    run_sub.font.color.rgb = MUTED_TEXT

    doc.add_paragraph() # Spacer

    # Helper function for Headers
    def add_custom_heading(text, level=1):
        h = doc.add_paragraph()
        run = h.add_run(text)
        run.font.name = "Arial"
        run.font.bold = True
        if level == 1:
            run.font.size = Pt(14)
            run.font.color.rgb = PRIMARY_COLOR
            h.paragraph_format.space_before = Pt(14)
            h.paragraph_format.space_after = Pt(4)
        elif level == 2:
            run.font.size = Pt(12)
            run.font.color.rgb = DARK_TEXT
            h.paragraph_format.space_before = Pt(10)
            h.paragraph_format.space_after = Pt(3)
        return h

    # Helper function for Code / Schema Blocks
    def add_code_block(code_text):
        tbl = doc.add_table(rows=1, cols=1)
        tbl.alignment = WD_TABLE_ALIGNMENT.CENTER
        cell = tbl.cell(0, 0)
        cell.width = Inches(6.5)
        set_cell_background(cell, CODE_BG)
        set_cell_margins(cell, top=140, bottom=140, left=180, right=180)

        p = cell.paragraphs[0]
        p.paragraph_format.space_before = Pt(2)
        p.paragraph_format.space_after = Pt(2)
        p.paragraph_format.line_spacing = 1.15
        run = p.add_run(code_text)
        run.font.name = "Consolas"
        run.font.size = Pt(8.5)
        run.font.color.rgb = DARK_TEXT

    # Helper function for standard paragraphs
    def add_body_p(text, bold_prefix=None):
        p = doc.add_paragraph()
        p.paragraph_format.space_after = Pt(4)
        p.paragraph_format.line_spacing = 1.15
        if bold_prefix:
            run_b = p.add_run(bold_prefix)
            run_b.font.name = "Arial"
            run_b.font.bold = True
            run_b.font.size = Pt(10)
            run_b.font.color.rgb = DARK_TEXT
        run = p.add_run(text)
        run.font.name = "Arial"
        run.font.size = Pt(10)
        run.font.color.rgb = DARK_TEXT
        return p

    # --- Section 1: Executive Overview & Fund Routing ---
    add_custom_heading("1. Executive Overview & Fund Settlement Mechanics", 1)
    add_body_p("This document details the architecture, data flows, persistence models, and webhook listeners implementing Safaricom Daraja M-Pesa Express (STK Push) on the Jenga P2P Hardware Marketplace platform.")
    
    add_custom_heading("Where Does the Money Go?", 2)
    add_body_p(
        "No actual fiat currency moves. Safaricom simulates balance deductions on the customer wallet and updates internal mock ledgers. You test end-to-end API round trips without financial exposure.",
        bold_prefix="• Sandbox Environment (ShortCode 174379): "
    )
    add_body_p(
        "Funds are instantly deducted from the buyer's Safaricom SIM mobile wallet and settled into the merchant's Lipa Na M-Pesa Organization Account (Paybill or Buy Goods Till). From Safaricom's Organization Management Portal (org.ke.mpesa.com), funds auto-sweep or disburse to your corporate Kenyan commercial bank account (e.g., Equity Bank, KCB, Standard Chartered) or are routed directly to suppliers and workshop vendors via B2C API calls.",
        bold_prefix="• Production (Live System): "
    )

    # --- Section 2: End-to-End System Architecture ---
    add_custom_heading("2. End-to-End System Architecture & Data Flow", 1)
    add_body_p("The payment lifecycle enforces strict asynchronous consistency. Orders are initialized in a PENDING_PAYMENT state and are solely committed when Safaricom transmits an authenticated webhook payload containing ResultCode = 0.")

    flowchart_ascii = (
        "+--------------------------------------------------------------------------------------------------+\n"
        "|                                    CUSTOMER CHECKOUT WORKFLOW                                    |\n"
        "+--------------------------------------------------------------------------------------------------+\n\n"
        " [ React Client ] \n"
        "   │ \n"
        "   │ 1. POST /api/orders/checkout (Cart, Idempotency-Key, Address, Phone: 2547XXXXXXXX)\n"
        "   ▼ \n"
        " [ OrderController ] \n"
        "   │ \n"
        "   ▼ \n"
        " [ CheckoutService ] ────► 2. Save Order (Status: PENDING_PAYMENT, Tracking: JNG-2026-XXXXX)\n"
        "   │                           │ \n"
        "   │                           ▼ \n"
        "   │                     [ MySQL: orders ] \n"
        "   │ \n"
        "   ▼ \n"
        " [ MpesaService ] ───────► 3. POST /oauth/v1/generate (Get Bearer Token)\n"
        "   │                           │ \n"
        "   │                           ▼ \n"
        "   │                      4. POST /mpesa/stkpush/v1/processrequest \n"
        "   │                         (PartyA, Shortcode: 174379, Password, CallbackURL: ngrok)\n"
        "   ▼ \n"
        " [ Safaricom Daraja ] ───► 5. SIM ToolKit (STK) Prompt pushed to buyer's handset\n"
        "   │ \n"
        "   ├───────────────────────────────────────────────┐ \n"
        "   │ (Polling loop every 3s)                       │ \n"
        "   ▼                                               │ \n"
        " [ React: GET /api/orders/track/{trackingNumber} ] │ 6. Customer enters M-Pesa PIN\n"
        "   ▲                                               │ \n"
        "   │                                               ▼ \n"
        "   │                                        [ Safaricom Gateway ] \n"
        "   │                                               │ \n"
        "   │                                               │ 7. POST CallBackURL (Async Webhook)\n"
        "   │                                               ▼ \n"
        "   │                                        [ ngrok Tunnel ] \n"
        "   │                                               │ (Forward to localhost:8080)\n"
        "   │                                               ▼ \n"
        "   │                                    [ PaymentWebhookController ] \n"
        "   │                                               │ \n"
        "   │                                               ▼ \n"
        "   │                                        [ CheckoutService ] \n"
        "   │                                               │ \n"
        "   │                                               ├─► 8. ResultCode == 0 ?\n"
        "   │                                               │      Deduct product stock quantities\n"
        "   │                                               │      Set OrderStatus = CONFIRMED\n"
        "   │                                               │      Set PaymentStatus = PAID\n"
        "   │                                               │      Set MpesaReceiptNumber (e.g. QHJ7XXXXX)\n"
        "   │                                               ▼ \n"
        "   │                                         [ MySQL DB ] \n"
        "   │                                               │ \n"
        "   └───────────────── 9. Status matches PAID ──────┘ \n"
        "                      Halts polling loop & renders Green Confirmation Card with Receipt Code"
    )
    add_code_block(flowchart_ascii)

    # --- Section 3: Backend Directory Structure ---
    add_custom_heading("3. Backend Directory & Package Structure", 1)
    add_body_p("The backend is structured according to domain-driven layer separation in Spring Boot:")

    tree_ascii = (
        "jenga/\n"
        "├── pom.xml\n"
        "├── src/\n"
        "│   ├── main/\n"
        "│   │   ├── java/\n"
        "│   │   │   └── com/\n"
        "│   │   │       └── hardware/\n"
        "│   │   │           └── jenga/\n"
        "│   │   │               ├── JengaApplication.java            # Spring Boot application bootstrap\n"
        "│   │   │               │\n"
        "│   │   │               ├── config/                          # Security & Web Configuration\n"
        "│   │   │               │   ├── JwtAuthenticationFilter.java # Stateless JWT token validation\n"
        "│   │   │               │   ├── JwtUtils.java                # Claims parsing & signature extraction\n"
        "│   │   │               │   └── WebSecurityConfig.java       # Whitelists webhooks, CORS & filters\n"
        "│   │   │               │\n"
        "│   │   │               ├── controller/                      # REST API Endpoints\n"
        "│   │   │               │   ├── AuthController.java          # Authentication & user profile endpoints\n"
        "│   │   │               │   ├── ProductController.java       # Catalog retrieval & product CRUD\n"
        "│   │   │               │   ├── OrderController.java         # /api/orders/checkout & tracking APIs\n"
        "│   │   │               │   └── PaymentWebhookController.java# /api/payments/mpesa/callback endpoint\n"
        "│   │   │               │\n"
        "│   │   │               ├── dto/                             # Data Transfer Objects\n"
        "│   │   │               │   ├── CheckoutDtos.java            # Immutable records for checkout & tracking\n"
        "│   │   │               │   └── MpesaCallbackDto.java        # Daraja STK callback JSON model\n"
        "│   │   │               │\n"
        "│   │   │               ├── entity/                          # JPA Database Entities\n"
        "│   │   │               │   ├── User.java                    # Buyers, sellers, and roles\n"
        "│   │   │               │   ├── Product.java                 # Catalog items & stock counters\n"
        "│   │   │               │   ├── Category.java                # Hardware categorizations\n"
        "│   │   │               │   ├── Order.java                   # Indexed tracking number, receipts, status\n"
        "│   │   │               │   └── OrderItem.java               # Line items attached to orders\n"
        "│   │   │               │\n"
        "│   │   │               ├── repository/                      # Spring Data JPA Interfaces\n"
        "│   │   │               │   ├── UserRepository.java\n"
        "│   │   │               │   ├── ProductRepository.java\n"
        "│   │   │               │   └── OrderRepository.java         # Lookup by tracking number & checkout request ID\n"
        "│   │   │               │\n"
        "│   │   │               └── service/                         # Domain Logic & External Integrations\n"
        "│   │   │                   ├── CheckoutService.java         # Transactional order commit, stock decrement\n"
        "│   │   │                   └── MpesaService.java            # Daraja OAuth token generator & STK prompt\n"
        "│   │   │\n"
        "│   │   └── resources/\n"
        "│   │       ├── application.properties                       # DB credentials, Daraja keys, ngrok URL\n"
        "│   │       └── static/\n"
        "│   │\n"
        "│   └── test/                                                # Unit & integration test cases\n"
        "└── uploads/                                                 # Statically served product images & media"
    )
    add_code_block(tree_ascii)

    # --- Section 4: Component Responsibilities Table ---
    add_custom_heading("4. Component Responsibilities & Layer Mapping", 1)
    
    table_data = [
        ("Component", "Layer", "Primary Architectural Responsibility"),
        ("Order.java", "Entity (JPA)", "Maintains order state with indexes on tracking_number, idempotency_key, and checkout_request_id. Allows guest checkout (buyer is nullable)."),
        ("OrderItem.java", "Entity (JPA)", "Maintains relational line items with fixed unit price snapshots at time of transaction."),
        ("MpesaService.java", "Service Layer", "Fetches Daraja bearer tokens, computes timestamped base64 passwords, formats MSISDN to 2547XXXXXXXX, and executes STK Push."),
        ("CheckoutService.java", "Service Layer", "Performs idempotency checks, verifies stock availability, orchestrates payment triggers, and confirms orders upon webhook validation."),
        ("PaymentWebhookController.java", "Controller", "Exposes POST /api/payments/mpesa/callback. Accepts incoming callbacks from Daraja and immediately responds with 200 OK acknowledgment."),
        ("WebSecurityConfig.java", "Security Config", "Disables CSRF for REST APIs, configures permissive CORS for ngrok domains, and permits unauthenticated access to webhooks."),
        ("Checkout.tsx", "React Frontend", "Manages local cart state, generates client idempotency keys, triggers orders, and polls order tracking status until confirmation.")
    ]

    tbl = doc.add_table(rows=len(table_data), cols=3)
    tbl.alignment = WD_TABLE_ALIGNMENT.CENTER

    col_widths = [Inches(1.8), Inches(1.2), Inches(3.5)]
    for row_idx, row in enumerate(tbl.rows):
        for col_idx, cell in enumerate(row.cells):
            cell.width = col_widths[col_idx]
            cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            set_cell_margins(cell, top=100, bottom=100, left=120, right=120)
            
            p = cell.paragraphs[0]
            p.paragraph_format.space_before = Pt(2)
            p.paragraph_format.space_after = Pt(2)
            run = p.add_run(table_data[row_idx][col_idx])
            run.font.name = "Arial"

            if row_idx == 0:
                set_cell_background(cell, HEADER_BG)
                run.font.bold = True
                run.font.size = Pt(9.5)
                run.font.color.rgb = RGBColor(255, 255, 255)
            else:
                set_cell_background(cell, "FFFFFF" if row_idx % 2 != 0 else "F8FAFC")
                run.font.size = Pt(8.5)
                run.font.color.rgb = DARK_TEXT

    doc.add_paragraph() # Spacer

    # --- Section 5: Database Schema Specification ---
    add_custom_heading("5. Database Schema & Indexing (MySQL: jenga_p2p_db)", 1)
    add_body_p("Optimized DDL ensuring ACID guarantees and sub-millisecond lookup latency during Safaricom callback ingestion:")

    sql_code = (
        "CREATE TABLE IF NOT EXISTS orders (\n"
        "    id BIGINT AUTO_INCREMENT PRIMARY KEY,\n"
        "    tracking_number VARCHAR(32) NOT NULL,\n"
        "    idempotency_key VARCHAR(64) NOT NULL,\n"
        "    user_id BIGINT NULL,\n"
        "    customer_name VARCHAR(255) NOT NULL,\n"
        "    customer_email VARCHAR(255) NOT NULL,\n"
        "    customer_phone VARCHAR(20) NOT NULL,\n"
        "    delivery_address TEXT NOT NULL,\n"
        "    subtotal_amount DECIMAL(12, 2) NOT NULL,\n"
        "    delivery_fee DECIMAL(12, 2) NOT NULL,\n"
        "    total_amount DECIMAL(12, 2) NOT NULL,\n"
        "    payment_method VARCHAR(20) NOT NULL,\n"
        "    payment_status VARCHAR(20) NOT NULL,\n"
        "    order_status VARCHAR(20) NOT NULL,\n"
        "    checkout_request_id VARCHAR(64) NULL,\n"
        "    mpesa_receipt_number VARCHAR(32) NULL,\n"
        "    created_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,\n"
        "    paid_at DATETIME NULL,\n"
        "    \n"
        "    CONSTRAINT fk_orders_user FOREIGN KEY (user_id) REFERENCES users(id) ON DELETE SET NULL,\n"
        "    CONSTRAINT uq_orders_tracking_number UNIQUE (tracking_number),\n"
        "    CONSTRAINT uq_orders_idempotency_key UNIQUE (idempotency_key),\n"
        "    INDEX idx_orders_tracking_number (tracking_number),\n"
        "    INDEX idx_orders_idempotency_key (idempotency_key),\n"
        "    INDEX idx_orders_checkout_request_id (checkout_request_id),\n"
        "    INDEX idx_orders_mpesa_receipt (mpesa_receipt_number)\n"
        ") ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;\n\n"
        "CREATE TABLE IF NOT EXISTS order_items (\n"
        "    id BIGINT AUTO_INCREMENT PRIMARY KEY,\n"
        "    order_id BIGINT NOT NULL,\n"
        "    product_id BIGINT NOT NULL,\n"
        "    quantity INT NOT NULL,\n"
        "    unit_price DECIMAL(12, 2) NOT NULL,\n"
        "    \n"
        "    CONSTRAINT fk_order_items_order FOREIGN KEY (order_id) REFERENCES orders(id) ON DELETE CASCADE,\n"
        "    CONSTRAINT fk_order_items_product FOREIGN KEY (product_id) REFERENCES products(id),\n"
        "    INDEX idx_order_items_order (order_id),\n"
        "    INDEX idx_order_items_product (product_id)\n"
        ") ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;"
    )
    add_code_block(sql_code)

    # --- Section 6: Production Go-Live Checklist ---
    add_custom_heading("6. Production Migration Checklist (Going Live)", 1)
    
    checklist = [
        ("Acquire Shortcode: ", "Obtain an official Safaricom Lipa Na M-Pesa Buy Goods Till or Paybill number via m-pesabusiness@safaricom.co.ke."),
        ("Portal Certification: ", "Sign into the Daraja Developer Portal and complete the 'Go Live' application by attaching registered company KYC documents."),
        ("Switch Endpoints: ", "Update MpesaService.java URL constants from https://sandbox.safaricom.co.ke to https://api.safaricom.co.ke."),
        ("Update Credentials: ", "Replace sandbox passkeys and consumer secrets with production equivalents issued by Safaricom."),
        ("Callback URL & SSL: ", "Switch the callback URL from your temporary ngrok domain to a production domain with valid SSL (e.g., https://api.jengamarketplace.co.ke/api/payments/mpesa/callback)."),
        ("IP Whitelisting: ", "Ensure firewall rules permit Safaricom's public IP ranges (196.201.214.*, 196.201.213.*, 196.201.212.*) to hit your webhook endpoint.")
    ]

    for prefix, body in checklist:
        add_body_p(body, bold_prefix=f"• {prefix}")

    output_filename = "Jenga_Marketplace_Mpesa_Order_Architecture.docx"
    doc.save(output_filename)
    print(f"Successfully generated: {output_filename}")

if __name__ == "__main__":
    create_document()

Successfully generated: Jenga_Marketplace_Mpesa_Order_Architecture.docx
